# Run Qwen 2.5 VL 7B on Google Colab
Make sure to set your runtime to use a GPU (**Runtime -> Change runtime type -> T4 GPU**).

## 1. Install Dependencies
We install `bitsandbytes` to load the model in 4-bit, which allows it to fit comfortably on the free T4 GPU (16GB VRAM).

In [ ]:
!pip install -q transformers accelerate bitsandbytes qwen-vl-utils huggingface_hub torchvision

## 2. Upload your image
Upload your `phase0_pairs.png` (from your `results/figures/` folder) to the Colab environment.

In [ ]:
from google.colab import files

print("Please upload phase0_pairs.png:")
uploaded = files.upload()
image_path = list(uploaded.keys())[0]
print(f"Uploaded {image_path}")

## 3. Load Model and Analyze Image

In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from transformers import BitsAndBytesConfig
from qwen_vl_utils import process_vision_info
import torch

# Use 4-bit quantization so the 7B model fits on the free Colab T4 GPU
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

print("Loading model (this might take a few minutes)...")
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-7B-Instruct", 
    device_map="auto",
    quantization_config=quantization_config
)
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct")
print("Model loaded successfully!")


In [ ]:
prompt = "Look at this grid of satellite images. The blue areas in the middle column represent water masks, and the red outlines in the right column show water boundaries. Can you describe the extent of flooding in these areas based on the images?"

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": f"file://{image_path}"},
            {"type": "text", "text": prompt},
        ],
    }
]

print("Processing inputs...")
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
image_inputs, video_inputs = process_vision_info(messages)

inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt"
).to(model.device)

print("Generating response...")
generated_ids = model.generate(**inputs, max_new_tokens=256)
generated_ids_trimmed = [
    out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)

print("\n--- Qwen2.5-VL Output ---")
print(output_text[0])